# Demo Backtest Viewer

Visualize outputs from `tools/demo_backtest.py`, including active management diagnostics:
- Equity/NAV and drawdown
- Portfolio vs benchmark returns
- Turnover and cost
- Fundamental Law metrics (IC, Breadth, TC, implied vs realized IR)
- Horizon diagnostics (IC/spread/hit for h1/h2/h4)
- Weights and orders

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Change this if your backtest output directory differs
BACKTEST_DIR = Path('../eval_results/backtest/demo_backtest')
assert BACKTEST_DIR.exists(), f'Backtest dir not found: {BACKTEST_DIR.resolve()}'

summary = json.loads((BACKTEST_DIR / 'summary.json').read_text())
equity = pd.read_csv(BACKTEST_DIR / 'equity_curve.csv')
weights = pd.read_csv(BACKTEST_DIR / 'weights_history.csv')
orders = pd.read_csv(BACKTEST_DIR / 'orders_history.csv')

equity['trade_date'] = pd.to_datetime(equity['trade_date'])
equity['next_date'] = pd.to_datetime(equity['next_date'])
weights['trade_date'] = pd.to_datetime(weights['trade_date'])
if 'trade_date' in orders.columns:
    orders['trade_date'] = pd.to_datetime(orders['trade_date'])

summary

In [ ]:
summary_df = pd.DataFrame([summary]).T
summary_df.columns = ['value']
summary_df

## Equity and Drawdown

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
equity.plot(x='trade_date', y='nav', ax=ax, legend=False, title='Equity Curve (NAV)')
ax.set_xlabel('trade_date')
ax.set_ylabel('nav')
plt.tight_layout()
plt.show()

In [ ]:
eq = equity[['trade_date', 'nav']].copy()
eq['running_max'] = eq['nav'].cummax()
eq['drawdown'] = eq['nav'] / eq['running_max'] - 1.0

fig, ax = plt.subplots(figsize=(10, 3))
eq.plot(x='trade_date', y='drawdown', ax=ax, legend=False, title='Drawdown')
ax.set_xlabel('trade_date')
ax.set_ylabel('drawdown')
plt.tight_layout()
plt.show()

eq[['trade_date', 'drawdown']].sort_values('drawdown').head(5)

## Returns and Turnover

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
equity.plot(x='trade_date', y='portfolio_return', ax=ax, label='portfolio_return', alpha=0.85)
equity.plot(x='trade_date', y='benchmark_return', ax=ax, label='benchmark_return', alpha=0.85)
ax.set_title('Periodic Returns')
ax.set_xlabel('trade_date')
ax.set_ylabel('return')
plt.tight_layout()
plt.show()

In [ ]:
turn_cols = [c for c in ['raw_turnover', 'executed_turnover', 'turnover_constraint_drag'] if c in equity.columns]
fig, ax = plt.subplots(figsize=(10, 4))
if turn_cols:
    equity.plot(x='trade_date', y=turn_cols, ax=ax, title='Turnover Decomposition')
else:
    equity.plot(x='trade_date', y='turnover', ax=ax, legend=False, title='Turnover by Rebalance')
ax.set_xlabel('trade_date')
ax.set_ylabel('turnover')
plt.tight_layout()
plt.show()

view_cols = [c for c in ['trade_date', 'turnover', 'raw_turnover', 'executed_turnover', 'turnover_constraint_drag', 'cost'] if c in equity.columns]
equity[view_cols].tail(10)

## Fundamental Law Diagnostics

In [ ]:
law_keys = [
    'average_ic',
    'average_breadth_proxy',
    'average_transfer_coefficient_proxy',
    'implied_information_ratio',
    'realized_active_information_ratio',
]
law = {k: summary.get(k) for k in law_keys}
pd.DataFrame([law]).T.rename(columns={0: 'value'})

In [ ]:
ir_df = pd.DataFrame({
    'type': ['implied_ir', 'realized_active_ir'],
    'value': [summary.get('implied_information_ratio'), summary.get('realized_active_information_ratio')]
})
fig, ax = plt.subplots(figsize=(6, 3))
ir_df.plot(kind='bar', x='type', y='value', ax=ax, legend=False, title='Implied vs Realized Active IR')
ax.set_ylabel('IR')
plt.tight_layout()
plt.show()
ir_df

## Horizon Diagnostics (h1/h2/h4)

In [ ]:
horizon_cols = [c for c in equity.columns if c.startswith('metric_ic_h') or c.startswith('metric_spread_h') or c.startswith('metric_hit_h')]
horizon_cols[:20], len(horizon_cols)

In [ ]:
ic_cols = [c for c in equity.columns if c.startswith('metric_ic_h')]
if ic_cols:
    fig, ax = plt.subplots(figsize=(10, 4))
    equity.plot(x='trade_date', y=ic_cols, ax=ax, title='Horizon IC Time Series')
    ax.set_xlabel('trade_date')
    ax.set_ylabel('IC')
    plt.tight_layout()
    plt.show()

    ic_summary = pd.DataFrame({
        'horizon': ic_cols,
        'mean_ic': [equity[c].mean() for c in ic_cols],
        'std_ic': [equity[c].std(ddof=0) for c in ic_cols],
    })
    ic_summary['ic_ir_proxy'] = ic_summary['mean_ic'] / ic_summary['std_ic']
    ic_summary
else:
    print('No horizon IC columns found.')

In [ ]:
spread_cols = [c for c in equity.columns if c.startswith('metric_spread_h')]
hit_cols = [c for c in equity.columns if c.startswith('metric_hit_h')]

if spread_cols:
    fig, ax = plt.subplots(figsize=(10, 4))
    equity.plot(x='trade_date', y=spread_cols, ax=ax, title='Top-Bottom Spread by Horizon')
    ax.set_xlabel('trade_date')
    ax.set_ylabel('spread')
    plt.tight_layout()
    plt.show()

if hit_cols:
    fig, ax = plt.subplots(figsize=(10, 3))
    equity.plot(x='trade_date', y=hit_cols, ax=ax, title='Top-Bottom Hit Rate Signal (0/1)')
    ax.set_xlabel('trade_date')
    ax.set_ylabel('hit')
    plt.tight_layout()
    plt.show()

horizon_rollup = {}
for c in spread_cols + hit_cols:
    horizon_rollup[c] = float(equity[c].mean())
pd.DataFrame([horizon_rollup]).T.rename(columns={0: 'average'})

## Weights and Orders

In [ ]:
weight_cols = [c for c in weights.columns if c != 'trade_date']
avg_weights = weights[weight_cols].mean().sort_values(ascending=False)
avg_weights.head(20)

In [ ]:
top_avg = avg_weights.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
top_avg.plot(kind='barh', ax=ax, title='Top 20 Average Weights')
ax.set_xlabel('average weight')
plt.tight_layout()
plt.show()

In [ ]:
orders.head(20)

In [ ]:
order_stats = {
    'total_orders': int(len(orders)),
    'total_estimated_cost': float(orders['estimated_cost'].sum()) if 'estimated_cost' in orders.columns else None,
    'buy_orders': int((orders['action'] == 'BUY').sum()) if 'action' in orders.columns else None,
    'sell_orders': int((orders['action'] == 'SELL').sum()) if 'action' in orders.columns else None,
}
order_stats